# HGP on PEMS-BAY (Kaggle)

Horizon-specific causal graphs (Part A) and HICP (Part B) on PEMS-BAY, 15 / 30 / 60 minute horizons.
One session builds the graphs once, then trains every (configuration, seed) pair listed in the settings cell and prints the
results table.

**Before running**
1. Session options: Accelerator = GPU (T4 or P100). Internet = On if the code is cloned from GitHub.
2. Add as inputs: a PEMS-BAY dataset holding `pems-bay.h5` and `adj_mx_bay.pkl`, and either this repository as a Kaggle
   dataset (a folder containing `hgp/`, `scripts/`) or set `REPO_URL` below.
3. Change only the settings cell. The training protocol (optimiser, learning rate, batch size, epochs, early stopping,
   split) is fixed in the code; only the method-specific settings below are meant to be tuned.

In [ ]:
# ---- settings
RUNS = [("none", 42), ("family", 42), ("g12_all", 42)]   # (configuration, seed); more: g3_all, shuffled, road, road_power
TAG = "base"                                              # names this setting; results and graphs go to folders with this tag

# method-specific hyperparameters (the only ones meant to be changed)
FDR = 0.3            # knockoff+ level q: expected share of false edges among the selected ones
GROUP_CORR = 0.5     # candidates with |correlation| above this form one knockoff group
MAX_SOURCES = 25     # at most this many sources per target
VAR_LAG = 3          # lags of the road-restricted VAR
VAR_RIDGE = 1e6      # ridge penalty of the VAR (scales with the data: raw speeds here)
HIDDEN = 64          # width of the temporal encoder
DROPOUT = 0.1

REPO_URL = ""        # e.g. "https://github.com/<user>/<repo>.git" when the code is not added as a Kaggle dataset
RUN_TESTS = True

## 1. Environment

In [ ]:
import glob
import os
import shutil
import subprocess
import sys

import torch

print(sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    print("warning: no GPU; PEMS-BAY (325 sensors) still runs on CPU but slowly")

## 2. Code and data

In [ ]:
WORK = "/kaggle/working"

hits = glob.glob("/kaggle/input/**/hgp/config.py", recursive=True)
if hits:
    CODE = os.path.dirname(os.path.dirname(hits[0]))
elif REPO_URL:
    CODE = f"{WORK}/hgp_code"
    shutil.rmtree(CODE, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
    assert os.path.isdir(f"{CODE}/hgp"), "the cloned repository has no hgp/ package at its top level"
else:
    raise SystemExit("HGP code not found: add the repository as a Kaggle dataset or set REPO_URL")
print("code:", CODE)

files = [p for p in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(p)]
traffic = [p for p in files if p.lower().endswith(".h5") and "pems" in os.path.basename(p).lower()]
adjacency = [p for p in files if p.lower().endswith(".pkl") and "adj" in os.path.basename(p).lower()]
assert traffic, "no PEMS-BAY .h5 file under /kaggle/input (expected pems-bay.h5)"
assert adjacency, "no adjacency .pkl under /kaggle/input (expected adj_mx_bay.pkl)"
TRAFFIC, ADJACENCY = traffic[0], adjacency[0]
print("traffic  :", TRAFFIC)
print("adjacency:", ADJACENCY)

In [ ]:
def run(cmd, show=True):
    print(" ".join(cmd))
    with subprocess.Popen(cmd, cwd=CODE, env=dict(os.environ, PYTHONUNBUFFERED="1"), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            if show and "Warning" not in line and "warnings.warn" not in line:
                print(line, end="")
    assert proc.returncode == 0, f"exited with {proc.returncode}"


if RUN_TESTS:
    run([sys.executable, "-m", "pytest", "tests/test_graphs.py", "tests/test_model.py", "tests/test_pems_bay.py", "-q",
         "-p", "no:cacheprovider", "-W", "ignore"])

## 3. Prepare the data
Chronological 70 / 10 / 20 split, zeros are missing speeds, short gaps are filled for the model inputs only. The printout shows the size of the candidate pool: if it covers most of the network, knockoff selection has little to work with.

In [ ]:
DATA = f"{WORK}/data/pems_bay.npz"
run([sys.executable, "scripts/prepare_data.py", "--dataset", "pems_bay", "--traffic", TRAFFIC, "--adjacency", ADJACENCY, "--out", DATA])

## 4. Part A: horizon-specific graphs (built once per setting)

In [ ]:
GRAPHS = f"{WORK}/graphs_{TAG}"
if all(os.path.exists(f"{GRAPHS}/G_h{h}.npz") for h in (3, 6, 12)):
    print("graphs already built:", GRAPHS)
else:
    run([sys.executable, "scripts/build_graphs.py", "--data", DATA, "--horizons", "3", "6", "12", "--fdr", str(FDR),
         "--group-corr", str(GROUP_CORR), "--max-sources", str(MAX_SOURCES), "--var-lag", str(VAR_LAG),
         "--var-ridge", str(VAR_RIDGE), "--out", GRAPHS])

## 5. Part B: train and evaluate
Each run writes `results.json`, `history.csv`, `config.json`, `checkpoints/best.pt` and `test_predictions_h{3,6,12}.npz`.

In [ ]:
RESULTS = f"{WORK}/results_{TAG}"
for config, seed in RUNS:
    if os.path.exists(f"{RESULTS}/{config}__seed{seed}/results.json") or os.path.exists(f"{RESULTS}/{config}_nograph__seed{seed}/results.json"):
        print("already done:", config, seed)
        continue
    run([sys.executable, "scripts/train.py", "--config", config, "--seed", str(seed), "--data", DATA, "--graphs", GRAPHS,
         "--results", RESULTS, "--hidden", str(HIDDEN), "--dropout", str(DROPOUT)])

## 6. Results
Mean ± standard deviation over seeds (one seed shows the value only). With two or more seeds the paired RMSE ratio `family / g12_all` (below 1 favours the horizon-specific graphs) is printed per seed with a 95% bootstrap interval over sensors.

In [ ]:
run([sys.executable, "scripts/summarise.py", "--results", RESULTS, "--metric", "rmse"])
run([sys.executable, "scripts/summarise.py", "--results", RESULTS, "--metric", "mae"])